[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-2_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 2: Priority-Based Construction Heuristic

### Purpose and scope

This notebook constructs vehicle routes for a Vehicle Routing Problem with Backhauls (VRPB). A vehicle starts at a depot, serves delivery customers, then serves pickup customers, and returns to the depot. The construction heuristic repeatedly chooses the feasible customer with the highest priority score. It completes the delivery phase before considering pickups on each route.

The inputs are customer demands, vehicle capacity, a depot-and-customer distance matrix, and priority weights. The outputs are an ordered list of routes, their distances, and their vehicle-load profiles. The method is a greedy heuristic: it produces a feasible construction when its assumptions are met, but it does not prove that the routes minimize total distance.

### Modeling assumptions

- Customer indices identify rows and columns in the distance matrix. Index `0` is the depot, indices `1` through `num_deliveries` are delivery customers, and the remaining customer indices are pickup customers.
- Demand values are nonnegative, each individual demand is no larger than vehicle capacity, and distances are nonnegative. The example uses a symmetric distance matrix; the route-cost calculation also works with an asymmetric matrix.
- All customers must be served at most once, and each constructed route visits deliveries before pickups. A route begins and ends at the depot.
- Vehicle capacity is interpreted as the physical load on board. The vehicle begins a route carrying that route's delivery goods. Deliveries reduce the load; after the delivery phase, pickups increase it. Thus, the delivery and pickup phases each respect the vehicle's capacity.
- The model does not include time windows, service times, vehicle-dependent costs, multiple depots, or restrictions on the number of vehicles beyond the `max_vehicles` limit.

### Construction procedure

For each route, the heuristic starts at the depot and repeatedly selects the feasible delivery with the greatest delivery priority. When no remaining delivery fits, it moves to the pickup phase and repeatedly selects the feasible pickup with the greatest pickup priority. Each chosen customer is removed from the unserved set. Route construction continues until every customer has been served or the vehicle limit is reached.

The priority scores combine demand, distance, available capacity, and—depending on customer type—urgency or proximity. A larger score is preferred. These scores guide the construction; they are not route costs and do not guarantee a globally shortest solution.

### Mathematical description

Let $N_D$ be the set of delivery customers, $N_P$ the set of pickup customers, and $N=N_D\cup N_P$. Let $d_{ij}$ be the distance from node $i$ to node $j$, $q_i$ the demand of customer $i$, and $Q$ the vehicle capacity. For a route $r=(0,i_1,\ldots,i_k,0)$, the distance is

$$
L(r)=d_{0i_1}+\sum_{t=1}^{k-1}d_{i_t i_{t+1}}+d_{i_k0}.
$$

The construction aims to produce a collection of routes $\mathcal{R}$ with small total distance $\sum_{r\in\mathcal{R}}L(r)$, subject to serving each customer exactly once, respecting capacity, and placing every delivery before every pickup on the same route. The heuristic uses these conditions to test feasibility and construct routes; it does not solve this minimization problem exactly.

At a construction step, let $U_D$ and $U_P$ be the unserved delivery and pickup sets. The chosen customer is the feasible candidate with maximum priority:

$$
i^*\in\underset{i\in F}{\operatorname{argmax}}\;P(i),
$$

where $F$ is the set of unserved customers that fit in the current phase's remaining capacity and $P(i)$ is the corresponding delivery or pickup priority. If $F$ is empty, the delivery phase ends and the pickup phase begins; if no pickup fits, the route ends. Ties are resolved by the order in which candidates appear in the unserved list.

In the code, `available_deliveries` and `available_pickups` represent the unserved sets, `calculate_delivery_priority` and `calculate_pickup_priority` compute $P(i)$, and `construct_single_route` carries out the two phases. `calculate_route_distance` evaluates $L(r)$. Repeatedly scanning the remaining customers gives a worst-case construction time of $O(n^2)$ for $n$ customers, excluding the cost of storing the distance matrix, which is $O(n^2)$.

### Interpreting results

Check that every customer is served exactly once, all route loads stay within capacity, and each mixed route serves deliveries before pickups. Route distance is the sum of its depot-to-customer, customer-to-customer, and customer-to-depot distances. A short computation time or a feasible route is not evidence of optimality; assessing solution quality requires a valid benchmark or a proven optimum for the same instance.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Use consistent plotting defaults so route and priority comparisons are easy to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for the priority-based VRPB construction heuristic.")

In [ ]:
@dataclass
class VRPBInstance:
    """Data for a vehicle routing problem with backhauls."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self):
        """Create customer index sets and validate the data used by the heuristic."""
        if self.num_deliveries < 0 or self.num_pickups < 0:
            raise ValueError("Customer counts must be nonnegative.")
        if len(self.delivery_demands) != self.num_deliveries:
            raise ValueError("The number of delivery demands must match num_deliveries.")
        if len(self.pickup_demands) != self.num_pickups:
            raise ValueError("The number of pickup demands must match num_pickups.")
        if self.capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")

        self.num_customers = self.num_deliveries + self.num_pickups
        expected_shape = (self.num_customers + 1, self.num_customers + 1)
        self.distance_matrix = np.asarray(self.distance_matrix, dtype=float)
        if self.distance_matrix.shape != expected_shape:
            raise ValueError(f"Distance matrix must have shape {expected_shape}.")
        if not np.all(np.isfinite(self.distance_matrix)):
            raise ValueError("Distance matrix entries must be finite.")
        if np.any(self.distance_matrix < 0):
            raise ValueError("Distances must be nonnegative.")

        all_demands = self.delivery_demands + self.pickup_demands
        if any(demand < 0 for demand in all_demands):
            raise ValueError("Customer demands must be nonnegative.")
        if any(demand > self.capacity for demand in all_demands):
            raise ValueError("Each customer demand must fit within vehicle capacity.")

        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class Route:
    """A vehicle route, with one load value recorded after each customer visit."""

    sequence: List[int]
    distance: float
    load_profile: List[float]
    route_type: str = "mixed"

    def __str__(self):
        route_str = " -> ".join(str(customer) for customer in self.sequence)
        return f"Route {self.route_type}: {route_str} | Distance: {self.distance:.2f}"


@dataclass
class PriorityWeights:
    """Weights used to combine the factors in each customer priority score."""

    delivery_demand_weight: float = 2.0
    delivery_distance_weight: float = 1.5
    delivery_capacity_weight: float = 1.0
    delivery_urgency_weight: float = 0.5
    pickup_demand_weight: float = 1.8
    pickup_distance_weight: float = 1.2
    pickup_capacity_weight: float = 1.5
    pickup_proximity_weight: float = 0.8


print("VRPB data structures and input checks are ready.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the small VRPB instance used to demonstrate the heuristic."""
    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    # Rows and columns use node indices: depot 0, deliveries 1-4, pickups 5-7.
    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ],
        dtype=float,
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(
    f"Created an instance with {instance.num_deliveries} deliveries "
    f"and {instance.num_pickups} pickups."
)
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
class PriorityBasedVRPB:
    """Greedy two-phase construction heuristic for the VRPB."""

    def __init__(
        self,
        instance: VRPBInstance,
        weights: Optional[PriorityWeights] = None,
    ):
        self.instance = instance
        self.weights = weights or PriorityWeights()
        self.routes: List[Route] = []
        self.unserved_customers: List[int] = []

    def calculate_delivery_priority(
        self,
        customer: int,
        current_position: int,
        current_load: float,
        remaining_capacity: float,
    ) -> float:
        """Score a feasible delivery; a higher score is selected first."""
        distance = self.instance.distance_matrix[current_position, customer]
        demand = self.instance.delivery_demands[customer - 1]
        capacity_factor = remaining_capacity / self.instance.capacity
        urgency = 1.0 / (1.0 + demand)
        distance_epsilon = 1e-6

        return (
            self.weights.delivery_demand_weight
            * demand
            / (distance + distance_epsilon)
            + self.weights.delivery_distance_weight
            * (1.0 / (distance + distance_epsilon))
            + self.weights.delivery_capacity_weight * capacity_factor
            + self.weights.delivery_urgency_weight * urgency
        )

    def calculate_pickup_priority(
        self,
        customer: int,
        current_position: int,
        current_load: float,
        remaining_capacity: float,
    ) -> float:
        """Score a feasible pickup; a higher score is selected first."""
        distance = self.instance.distance_matrix[current_position, customer]
        demand_index = customer - self.instance.num_deliveries - 1
        demand = self.instance.pickup_demands[demand_index]
        capacity_factor = remaining_capacity / self.instance.capacity
        proximity_bonus = 1.0 / (1.0 + distance)
        distance_epsilon = 1e-6

        return (
            self.weights.pickup_demand_weight
            * demand
            / (distance + distance_epsilon)
            + self.weights.pickup_distance_weight
            * (1.0 / (distance + distance_epsilon))
            + self.weights.pickup_capacity_weight * capacity_factor
            + self.weights.pickup_proximity_weight * proximity_bonus
        )

    def calculate_route_distance(self, route_sequence: List[int]) -> float:
        """Sum the travel distances from depot through the route and back."""
        if not route_sequence:
            return 0.0

        total_distance = self.instance.distance_matrix[0, route_sequence[0]]
        for first, second in zip(route_sequence, route_sequence[1:]):
            total_distance += self.instance.distance_matrix[first, second]
        total_distance += self.instance.distance_matrix[route_sequence[-1], 0]
        return float(total_distance)

    def calculate_load_profile(self, route_sequence: List[int]) -> List[float]:
        """Record the physical vehicle load after each customer visit."""
        initial_delivery_load = sum(
            self.instance.delivery_demands[customer - 1]
            for customer in route_sequence
            if customer in self.instance.delivery_indices
        )
        current_load = float(initial_delivery_load)
        load_profile = []

        for customer in route_sequence:
            if customer in self.instance.delivery_indices:
                current_load -= self.instance.delivery_demands[customer - 1]
            else:
                demand_index = customer - self.instance.num_deliveries - 1
                current_load += self.instance.pickup_demands[demand_index]
            load_profile.append(current_load)

        return load_profile

    def construct_single_route(
        self,
        available_deliveries: List[int],
        available_pickups: List[int],
    ) -> Route:
        """Build one route, completing its delivery phase before its pickup phase."""
        route_sequence: List[int] = []
        current_position = 0
        delivery_load = 0.0

        # Delivery capacity is checked against the total delivery goods assigned to this route.
        while available_deliveries:
            remaining_capacity = self.instance.capacity - delivery_load
            feasible_deliveries = [
                customer
                for customer in available_deliveries
                if self.instance.delivery_demands[customer - 1] <= remaining_capacity
            ]
            if not feasible_deliveries:
                break

            best_customer = max(
                feasible_deliveries,
                key=lambda customer: self.calculate_delivery_priority(
                    customer,
                    current_position,
                    delivery_load,
                    remaining_capacity,
                ),
            )
            route_sequence.append(best_customer)
            delivery_load += self.instance.delivery_demands[best_customer - 1]
            current_position = best_customer
            available_deliveries.remove(best_customer)

        # All delivery goods have been unloaded before pickups begin, so pickup capacity is available.
        pickup_load = 0.0
        while available_pickups:
            remaining_capacity = self.instance.capacity - pickup_load
            feasible_pickups = [
                customer
                for customer in available_pickups
                if self.instance.pickup_demands[
                    customer - self.instance.num_deliveries - 1
                ]
                <= remaining_capacity
            ]
            if not feasible_pickups:
                break

            best_customer = max(
                feasible_pickups,
                key=lambda customer: self.calculate_pickup_priority(
                    customer,
                    current_position,
                    pickup_load,
                    remaining_capacity,
                ),
            )
            route_sequence.append(best_customer)
            demand_index = best_customer - self.instance.num_deliveries - 1
            pickup_load += self.instance.pickup_demands[demand_index]
            current_position = best_customer
            available_pickups.remove(best_customer)

        distance = self.calculate_route_distance(route_sequence)
        load_profile = self.calculate_load_profile(route_sequence)
        has_deliveries = any(
            customer in self.instance.delivery_indices
            for customer in route_sequence
        )
        has_pickups = any(
            customer in self.instance.pickup_indices
            for customer in route_sequence
        )

        if has_deliveries and has_pickups:
            route_type = "mixed"
        elif has_deliveries:
            route_type = "delivery"
        else:
            route_type = "pickup"

        return Route(
            sequence=route_sequence,
            distance=distance,
            load_profile=load_profile,
            route_type=route_type,
        )

    def solve(self, max_vehicles: int = 10) -> List[Route]:
        """Construct routes until all customers are served or the vehicle limit is reached."""
        if max_vehicles < 0:
            raise ValueError("max_vehicles must be nonnegative.")

        print("Starting priority-based VRPB construction.")
        available_deliveries = self.instance.delivery_indices.copy()
        available_pickups = self.instance.pickup_indices.copy()
        self.routes = []
        vehicle_count = 0

        while (
            (available_deliveries or available_pickups)
            and vehicle_count < max_vehicles
        ):
            route = self.construct_single_route(
                available_deliveries.copy(),
                available_pickups.copy(),
            )
            if not route.sequence:
                break

            self.routes.append(route)
            vehicle_count += 1
            served_customers = set(route.sequence)
            available_deliveries = [
                customer
                for customer in available_deliveries
                if customer not in served_customers
            ]
            available_pickups = [
                customer
                for customer in available_pickups
                if customer not in served_customers
            ]
            print(
                f"  Route {vehicle_count}: {route.sequence} "
                f"(distance: {route.distance:.2f})"
            )

        self.unserved_customers = available_deliveries + available_pickups
        if self.unserved_customers:
            print(
                f"Vehicle limit reached; {len(self.unserved_customers)} "
                "customer(s) remain unserved."
            )
        return self.routes


print("Priority-based VRPB solver is ready.")

In [ ]:
solver = PriorityBasedVRPB(instance)
start_time = time.perf_counter()
routes = solver.solve(max_vehicles=5)
computation_time = time.perf_counter() - start_time

total_distance = sum(route.distance for route in routes)
total_customers = sum(len(route.sequence) for route in routes)

print("\nPriority-based heuristic results")
print("=" * 48)
print(f"Number of routes: {len(routes)}")
print(f"Total route distance: {total_distance:.2f}")
print(f"Customers served: {total_customers} of {instance.num_customers}")
print(f"Computation time: {computation_time * 1000:.3f} ms")

for route_number, route in enumerate(routes, start=1):
    full_sequence = [0] + route.sequence + [0]
    print(f"\nRoute {route_number} ({route.route_type})")
    print("Sequence: " + " -> ".join(map(str, full_sequence)))
    print(f"Distance: {route.distance:.2f}")
    print(f"Customers: {len(route.sequence)}")
    print(f"Load after each customer: {[round(load, 2) for load in route.load_profile]}")

    delivery_positions = [
        position
        for position, customer in enumerate(route.sequence)
        if customer in instance.delivery_indices
    ]
    pickup_positions = [
        position
        for position, customer in enumerate(route.sequence)
        if customer in instance.pickup_indices
    ]
    precedence_ok = (
        not delivery_positions
        or not pickup_positions
        or max(delivery_positions) < min(pickup_positions)
    )
    print(f"Delivery-before-pickup precedence: {precedence_ok}")

all_served = sorted(customer for route in routes for customer in route.sequence)
assert all_served == instance.all_customer_indices, (
    "The example should serve every customer exactly once."
)
assert not solver.unserved_customers
assert np.isclose(total_distance, sum(route.distance for route in routes))
for route in routes:
    assert len(route.sequence) == len(set(route.sequence))
    assert all(load <= instance.capacity + 1e-9 for load in route.load_profile)
    assert all(load >= -1e-9 for load in route.load_profile)
    assert route.distance == solver.calculate_route_distance(route.sequence)

print("\nRoute coverage, precedence, capacity, and distance checks passed.")

In [ ]:
def visualize_priority_solution(
    routes: List[Route],
    instance: VRPBInstance,
    solver: PriorityBasedVRPB,
):
    """Plot route distances, selected-customer priorities, and load profiles."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 11)
    )
    fig.suptitle("Priority-Based VRPB Route Analysis", fontsize=15, weight="bold")

    # Route distances show how the constructed travel cost is distributed across vehicles.
    route_numbers = np.arange(1, len(routes) + 1)
    route_distances = [route.distance for route in routes]
    if routes:
        sns.barplot(
            x=route_numbers,
            y=route_distances,
            hue=[route.route_type for route in routes],
            ax=ax1,
            dodge=False,
        )
    ax1.set_title("Distance by Route")
    ax1.set_xlabel("Route number")
    ax1.set_ylabel("Distance (matrix units)")
    if routes:
        ax1.legend(title="Route type")

    # Recompute each chosen customer's score at the state in which it was selected.
    if routes:
        first_route = routes[0]
        priorities = []
        customer_types = []
        current_position = 0
        delivery_load = 0.0
        pickup_load = 0.0
        pickup_phase = False

        for customer in first_route.sequence:
            if customer in instance.delivery_indices:
                remaining_capacity = instance.capacity - delivery_load
                priority = solver.calculate_delivery_priority(
                    customer,
                    current_position,
                    delivery_load,
                    remaining_capacity,
                )
                delivery_load += instance.delivery_demands[customer - 1]
                customer_types.append("Delivery")
            else:
                pickup_phase = True
                remaining_capacity = instance.capacity - pickup_load
                priority = solver.calculate_pickup_priority(
                    customer,
                    current_position,
                    pickup_load,
                    remaining_capacity,
                )
                demand_index = customer - instance.num_deliveries - 1
                pickup_load += instance.pickup_demands[demand_index]
                customer_types.append("Pickup")

            priorities.append(priority)
            current_position = customer

        positions = np.arange(1, len(first_route.sequence) + 1)
        colors = [
            "tab:blue" if customer_type == "Delivery" else "tab:orange"
            for customer_type in customer_types
        ]
        ax2.bar(positions, priorities, color=colors, alpha=0.8)
        ax2.set_xlabel("Visit position in the first route")
        ax2.set_ylabel("Priority score")
        ax2.set_title("Selected-Customer Priorities")
        ax2.grid(axis="y", alpha=0.3)
        ax2.legend(
            handles=[
                plt.Rectangle((0, 0), 1, 1, color="tab:blue", label="Delivery"),
                plt.Rectangle((0, 0), 1, 1, color="tab:orange", label="Pickup"),
            ],
            title="Customer type",
        )
    else:
        ax2.set_title("Selected-Customer Priorities (no routes)")

    # Each load value is recorded after visiting its corresponding customer.
    for route_number, route in enumerate(routes, start=1):
        visit_positions = np.arange(1, len(route.load_profile) + 1)
        ax3.plot(
            visit_positions,
            route.load_profile,
            marker="o",
            label=f"Route {route_number}",
        )
    ax3.axhline(
        instance.capacity,
        color="red",
        linestyle="--",
        label=f"Capacity ({instance.capacity})",
    )
    ax3.set_xlabel("Visit position in route")
    ax3.set_ylabel("Load after visit (demand units)")
    ax3.set_title("Vehicle Load Profiles")
    ax3.legend()
    ax3.grid(alpha=0.3)

    total_distance = sum(route.distance for route in routes)
    total_customers = sum(len(route.sequence) for route in routes)
    metrics = ["Routes", "Total distance", "Customers served"]
    values = [len(routes), total_distance, total_customers]
    ax4.bar(metrics, values, color="tab:green", alpha=0.8)
    ax4.set_title("Constructed Solution Summary")
    ax4.set_ylabel("Value")
    ax4.tick_params(axis="x", rotation=15)
    ax4.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_priority_solution(routes, instance, solver)

In [ ]:
def analyze_heuristic_performance(
    routes: List[Route],
    instance: VRPBInstance,
    solver: PriorityBasedVRPB,
):
    """Check feasibility and summarize construction cost and route balance."""
    print("Heuristic performance analysis")
    print("=" * 48)

    all_constraints_satisfied = not solver.unserved_customers
    visited_customers = []

    print("\nConstraint checks")
    for route_number, route in enumerate(routes, start=1):
        visited_customers.extend(route.sequence)
        delivery_positions = [
            position
            for position, customer in enumerate(route.sequence)
            if customer in instance.delivery_indices
        ]
        pickup_positions = [
            position
            for position, customer in enumerate(route.sequence)
            if customer in instance.pickup_indices
        ]
        precedence_ok = (
            not delivery_positions
            or not pickup_positions
            or max(delivery_positions) < min(pickup_positions)
        )
        capacity_ok = all(
            load <= instance.capacity + 1e-9 and load >= -1e-9
            for load in route.load_profile
        )
        visitation_ok = len(route.sequence) == len(set(route.sequence))
        route_ok = precedence_ok and capacity_ok and visitation_ok
        all_constraints_satisfied = all_constraints_satisfied and route_ok

        maximum_load = max(route.load_profile, default=0.0)
        print(f"\nRoute {route_number}:")
        print(f"  Delivery-before-pickup precedence: {precedence_ok}")
        print(
            f"  Capacity: {capacity_ok} "
            f"(maximum load {maximum_load:.2f} / {instance.capacity})"
        )
        print(f"  No repeated customer within route: {visitation_ok}")

    coverage_ok = (
        sorted(visited_customers) == instance.all_customer_indices
        and len(visited_customers) == len(set(visited_customers))
    )
    all_constraints_satisfied = all_constraints_satisfied and coverage_ok
    print(f"\nEvery customer served exactly once: {coverage_ok}")
    print(f"All checked constraints satisfied: {all_constraints_satisfied}")

    print("\nConstruction and distance summary")
    total_distance = sum(route.distance for route in routes)
    print(f"Customers in instance: {instance.num_customers}")
    print(f"Customers served: {len(visited_customers)}")
    print(f"Routes constructed: {len(routes)}")
    print(f"Total route distance: {total_distance:.2f}")
    print(
        "Construction-time bound: O(n^2) candidate evaluations "
        "in the worst case for n customers."
    )
    print(
        "This bound describes the construction procedure; it is not "
        "a measured runtime or a guarantee of solution quality."
    )

    if routes:
        distances = [route.distance for route in routes]
        customer_counts = [len(route.sequence) for route in routes]
        print("\nRoute balance")
        print(f"Route-distance standard deviation: {np.std(distances):.2f}")
        print(
            "Customers-per-route standard deviation: "
            f"{np.std(customer_counts):.2f}"
        )

    print("\nPriority weights")
    print(f"Delivery demand weight: {solver.weights.delivery_demand_weight}")
    print(f"Delivery distance weight: {solver.weights.delivery_distance_weight}")
    print(f"Delivery capacity weight: {solver.weights.delivery_capacity_weight}")
    print(f"Delivery urgency weight: {solver.weights.delivery_urgency_weight}")
    print(f"Pickup demand weight: {solver.weights.pickup_demand_weight}")
    print(f"Pickup distance weight: {solver.weights.pickup_distance_weight}")
    print(f"Pickup capacity weight: {solver.weights.pickup_capacity_weight}")
    print(f"Pickup proximity weight: {solver.weights.pickup_proximity_weight}")

    return all_constraints_satisfied


constraints_satisfied = analyze_heuristic_performance(routes, instance, solver)
assert constraints_satisfied, "The constructed example must satisfy its stated constraints."

In [ ]:
def test_scalability():
    """Run the same construction heuristic on reproducible synthetic instances."""
    print("Scalability experiments")
    print("=" * 48)

    test_sizes = [
        (10, 5, "Small"),
        (20, 10, "Medium"),
        (30, 15, "Large"),
    ]

    for num_deliveries, num_pickups, size_name in test_sizes:
        total_customers = num_deliveries + num_pickups
        print(f"\n{size_name} instance ({total_customers} customers)")

        # Reset the supplied seed for reproducible synthetic demand and distance data.
        np.random.seed(42)
        delivery_demands = np.random.randint(1, 8, num_deliveries).tolist()
        pickup_demands = np.random.randint(1, 6, num_pickups).tolist()
        capacity = 20

        distance_matrix = np.random.randint(
            2, 15, size=(total_customers + 1, total_customers + 1)
        )
        distance_matrix = (distance_matrix + distance_matrix.T) // 2
        np.fill_diagonal(distance_matrix, 0)

        test_instance = VRPBInstance(
            num_deliveries=num_deliveries,
            num_pickups=num_pickups,
            delivery_demands=delivery_demands,
            pickup_demands=pickup_demands,
            capacity=capacity,
            distance_matrix=distance_matrix,
        )
        test_solver = PriorityBasedVRPB(test_instance)
        start_time = time.perf_counter()
        test_routes = test_solver.solve(max_vehicles=10)
        elapsed_time = time.perf_counter() - start_time

        customers_served = sum(len(route.sequence) for route in test_routes)
        total_distance = sum(route.distance for route in test_routes)
        print(f"Elapsed construction time: {elapsed_time * 1000:.3f} ms")
        print(f"Routes constructed: {len(test_routes)}")
        print(f"Customers served: {customers_served} of {total_customers}")
        print(f"Total route distance: {total_distance:.2f}")
        if test_solver.unserved_customers:
            print(f"Unserved customers: {test_solver.unserved_customers}")
        else:
            print("All customers were served within the vehicle limit.")


test_scalability()

## Summary and conclusions

The notebook uses a two-phase, priority-based construction heuristic for the VRPB. Within each route, feasible deliveries are selected greedily until none can be added; pickups are then selected greedily using the capacity available after delivery goods have been unloaded. Route cost is calculated from the distance matrix, and the reported load profile tracks physical onboard demand after each customer visit.

The feasibility checks cover customer coverage, route-level uniqueness, delivery-before-pickup precedence, and vehicle capacity. A route set that passes these checks is feasible for the assumptions stated here. The `max_vehicles` limit can leave customers unserved, which the solver reports explicitly.

The priority weights shape which feasible customer is selected at each step. Because the method is greedy, the local choice with the largest score need not lead to the minimum total distance across all routes. The distance and runtime measurements describe the constructed solution and the particular run; they are not an optimality certificate or a general performance guarantee. A genuine optimality gap can only be reported when a valid optimal benchmark for the same instance is available.

The synthetic scalability experiments illustrate how the same construction procedure behaves on larger generated instances. Their distance matrices are generated data, not geographic travel networks, and measured runtimes depend on the execution environment. The worst-case construction work grows quadratically with the number of customers because each selection may scan the remaining candidates. The method does not account for time windows, service durations, or other operational constraints omitted from the model.